In [ ]:
# 1. 安裝必要的套件
!pip install -q transformers datasets timm huggingface_hub evaluate

In [ ]:
import os
import time
import torch
import requests
import PIL.Image
from tqdm import tqdm
from google.colab import widgets
from IPython.display import display, clear_output
from transformers import pipeline, AutoModelForImageClassification, AutoImageProcessor, TrainingArguments, Trainer
from datasets import Dataset, Features, ClassLabel, Image as DatasetImage
import numpy as np

# 設定
DATASET_URL = "https://github.com/szeu/hk-traffic-dataset/archive/refs/heads/main.zip"
IMAGE_DIR = "traffic_images"
LABELS = ['heavy-traffic', 'low-traffic', 'medium-traffic', 'no-traffic', 'signal-loss']
ITERATION_SIZE = 200 # 每次標注 200 張影像後進行一次訓練 (共約 4 輪)

# 2. 下載並解壓縮數據集
if not os.path.exists(IMAGE_DIR):
    !wget {DATASET_URL} -O main.zip
    !unzip -q main.zip
    !mkdir -p {IMAGE_DIR}
    # 搜尋所有 jpg/png 並移動到統一資料夾
    !find hk-traffic-dataset-main -name "*.jpg" -exec mv {} {IMAGE_DIR} \;
    !find hk-traffic-dataset-main -name "*.png" -exec mv {} {IMAGE_DIR} \;

all_image_paths = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))]
print(f"找到 {len(all_image_paths)} 張影像。")

# 3. 載入初步分類模型 (prithivMLmods/Traffic-Density-Classification)
print("正在載入初步分類模型...")
pre_model = pipeline("image-classification", model="prithivMLmods/Traffic-Density-Classification", device=0)

# 4. 互動式標注函式
def label_images(image_list):
    labeled_data = []
    for img_path in image_list:
        clear_output(wait=True)
        img = PIL.Image.open(img_path)
        display(img.resize((400, 300)))

        # 初步預測
        pred = pre_model(img)[0]['label'].lower()
        # 映射標籤 (若名稱不完全相符則需手動修正)
        mapped_pred = pred if pred in LABELS else "未匹配"

        print(f"路徑: {img_path}")
        print(f"預測標籤: {mapped_pred}")
        print(f"標籤選項: 0:{LABELS[0]}, 1:{LABELS[1]}, 2:{LABELS[2]}, 3:{LABELS[3]}, 4:{LABELS[4]}")
        time.sleep(.5)
        user_input = input("輸入數字選擇標籤，或直接按回車(Enter)接受預測標籤: \n").strip()

        if user_input == "" and mapped_pred in LABELS:
            final_label = mapped_pred
        elif user_input in ["0", "1", "2", "3", "4"]:
            final_label = LABELS[int(user_input)]
        else:
            final_label = "low-traffic" # 預設或錯誤處理

        labeled_data.append({"image": img_path, "label": final_label})
    return labeled_data

# 5. 微調 DeiT 模型函式
def train_deit(labeled_results, iteration_num):
    print(f"\n--- 開始第 {iteration_num} 輪 DeiT 微調 (數據量: {len(labeled_results)}) ---")

    # 1. 準備 Processor
    model_id = "facebook/deit-base-distilled-patch16-224"
    processor = AutoImageProcessor.from_pretrained(model_id)

    # 2. 手動預處理所有影像 (避免 JpegImageFile 錯誤)
    pixel_values = []
    labels = []

    print("正在預處理影像...")
    for item in labeled_results:
        img_path = item['image']
        label_str = item['label']

        # 讀取並轉換
        img = PIL.Image.open(img_path).convert("RGB")
        inputs = processor(img, return_tensors="pt")

        # 關鍵修改：轉為 Numpy 並存儲
        pixel_values.append(inputs.pixel_values.squeeze(0).numpy())
        labels.append(LABELS.index(label_str))

    # 3. 建立數據集 (使用 Numpy Array)
    full_ds = Dataset.from_dict({
        "pixel_values": np.array(pixel_values, dtype=np.float32),
        "label": labels  # 修正處：從 labels 改為 label
    })

    # 4. 分割訓練與測試集
    ds_split = full_ds.train_test_split(test_size=0.2, seed=42)
    train_ds = ds_split['train']
    test_ds = ds_split['test']

    # 5. 載入模型
    model = AutoModelForImageClassification.from_pretrained(
        model_id,
        num_labels=len(LABELS),
        id2label={i: l for i, l in enumerate(LABELS)},
        label2id={l: i for i, l in enumerate(LABELS)},
        ignore_mismatched_sizes=True
    ).to("cuda")

    # 6. 設定訓練參數
    training_args = TrainingArguments(
        output_dir=f"./deit-traffic-it{iteration_num}",
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=5e-5,
        per_device_train_batch_size=8,
        num_train_epochs=10,
        logging_steps=10,
        report_to="none",
        # 移除 label_names 參數，改用下方的自定義 Trainer 處理
    )

     # 7. 建立自定義 Trainer 手動計算 Loss
    class DeiTTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            # 提取標籤並從 inputs 中移除
            if "labels" in inputs:
                labels = inputs.pop("labels")
            elif "label" in inputs:
                labels = inputs.pop("label")
            else:
                labels = None

            # 取得模型輸出
            outputs = model(**inputs)

            # DeiT Distilled 輸出包含 logits (通常是 cls_logits 和 distillation_logits 的平均或組合)
            logits = outputs.logits

            # 手動計算 Cross Entropy Loss
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        # 這裡 eval_pred.predictions 會包含所有 logits
        # 對於 DeiT Distilled，通常 index 0 是最終可以用來分類的 logits
        predictions, labels = eval_pred
        if isinstance(predictions, (list, tuple)):
          predictions = predictions[0]

        preds = np.argmax(predictions, axis=-1)
        return {"accuracy": (preds == labels).astype(np.float32).mean().item()}

    # 使用自定義的 DeiTTrainer 啟動
    trainer = DeiTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    accuracy = metrics.get('eval_accuracy') or metrics.get('accuracy') or 0.0

    print(f"\n✅ 第 {iteration_num} 輪結果: Accuracy = {accuracy:.4f}")
    print(f"完整指標: {metrics}")

    return model

# 6. 主執行迴圈 (疊代標注與訓練)
total_labeled = []
for i in range(0, len(all_image_paths), ITERATION_SIZE):
    batch = all_image_paths[i : i + ITERATION_SIZE]
    iteration_num = (i // ITERATION_SIZE) + 1

    print(f"\n現在開始第 {iteration_num} 階段標注...")
    new_labeled = label_images(batch)
    total_labeled.extend(new_labeled)

    # 進行訓練
    current_model = train_deit(total_labeled, iteration_num)

    # 儲存目前的標注結果為 CSV
    import pandas as pd
    pd.DataFrame(total_labeled).to_csv("labeled_traffic_results.csv", index=False)

    input("按下 Enter 鍵開始下一階段標注...\n")

print("所有標注與訓練已完成。結果儲存在 labeled_traffic_results.csv")
